# Simple comparison with `confseq`

This notebook compares:

- `mapie.exchangeability_testing.confidence_bounds.GammaExponentialMixtureBound`
- `confseq.boundaries.gamma_exponential_mixture_bound`

It assumes `confseq` is available in the current Python environment.
If it is not installed yet, install it first in whatever environment you use
for notebooks.


In [1]:
import numpy as np

from mapie.exchangeability_testing.confidence_bounds import (
    GammaExponentialMixtureBound,
)

try:
    from confseq.boundaries import gamma_exponential_mixture_bound
except ImportError as exc:
    raise ImportError(
        "This notebook requires `confseq`. Install it in your notebook environment first."
    ) from exc


In [2]:
v_opt = 5.0
c = 1.0
alpha_opt = 0.025
alpha = 0.025

local_bound = GammaExponentialMixtureBound(
    v_opt=v_opt,
    c=c,
    alpha_opt=alpha_opt,
)

v_values = np.array([1e-6, 1e-3, 0.1, 0.5, 1.0, 2.5, 10.0, 100.0])


In [3]:
rows = []
for v in v_values:
    local_value = local_bound.bound(v=v, alpha=alpha)
    confseq_value = gamma_exponential_mixture_bound(
        v=v,
        alpha=alpha,
        v_opt=v_opt,
        c=c,
        alpha_opt=alpha_opt,
    )
    rows.append(
        {
            "v": v,
            "mapie": local_value,
            "confseq": confseq_value,
            "abs_diff": abs(local_value - confseq_value),
        }
    )

for row in rows:
    print(
        f"v={row['v']:<8g} "
        f"mapie={row['mapie']:.12f} "
        f"confseq={row['confseq']:.12f} "
        f"abs_diff={row['abs_diff']:.3e}"
    )


v=1e-06    mapie=4.617479044706 confseq=4.617479044704 abs_diff=1.322e-12
v=0.001    mapie=4.619989754961 confseq=4.619989754964 abs_diff=2.315e-12
v=0.1      mapie=4.857028191875 confseq=4.857028191877 abs_diff=2.548e-12
v=0.5      mapie=5.651337103396 confseq=5.651337103398 abs_diff=1.819e-12
v=1        mapie=6.437874798359 confseq=6.437874798361 abs_diff=1.819e-12
v=2.5      mapie=8.210269135996 confseq=8.210269135993 abs_diff=2.274e-12
v=10       mapie=13.544187794419 confseq=13.544187794423 abs_diff=4.547e-12
v=100      mapie=38.647569891793 confseq=38.647569891805 abs_diff=1.137e-11


In [4]:
max_abs_diff = max(row["abs_diff"] for row in rows)
print("max_abs_diff =", max_abs_diff)


max_abs_diff = 1.1368683772161603e-11


## Extended checks

The cells below broaden the comparison in three ways:

- multiple choices of `v_opt`, `c`, `alpha_opt`, and `alpha`,
- a randomized sweep over positive `v`,
- an explicit `v = 0` check.

For positive `v`, the two implementations should agree up to numerical solver tolerance.

In [5]:
parameter_sets = [
    {"v_opt": 0.5, "c": 0.5, "alpha_opt": 0.01, "alpha": 0.01},
    {"v_opt": 5.0, "c": 1.0, "alpha_opt": 0.025, "alpha": 0.025},
    {"v_opt": 20.0, "c": 2.0, "alpha_opt": 0.05, "alpha": 0.05},
]

v_grid = np.array([1e-9, 1e-6, 1e-3, 0.1, 1.0, 10.0, 100.0])

grid_rows = []
for params in parameter_sets:
    local_bound = GammaExponentialMixtureBound(
        v_opt=params["v_opt"],
        c=params["c"],
        alpha_opt=params["alpha_opt"],
    )
    for v in v_grid:
        mapie_value = local_bound.bound(v=v, alpha=params["alpha"])
        confseq_value = gamma_exponential_mixture_bound(
            v=v,
            alpha=params["alpha"],
            v_opt=params["v_opt"],
            c=params["c"],
            alpha_opt=params["alpha_opt"],
        )
        grid_rows.append(
            {
                **params,
                "v": v,
                "mapie": mapie_value,
                "confseq": confseq_value,
                "abs_diff": abs(mapie_value - confseq_value),
            }
        )

for row in grid_rows:
    print(
        f"v_opt={row['v_opt']:<5g} c={row['c']:<4g} alpha_opt={row['alpha_opt']:<6g} "
        f"alpha={row['alpha']:<6g} v={row['v']:<8g} abs_diff={row['abs_diff']:.3e}"
    )

print("max_grid_abs_diff =", max(row["abs_diff"] for row in grid_rows))

v_opt=0.5   c=0.5  alpha_opt=0.01   alpha=0.01   v=1e-09    abs_diff=2.481e-12
v_opt=0.5   c=0.5  alpha_opt=0.01   alpha=0.01   v=1e-06    abs_diff=1.500e-12
v_opt=0.5   c=0.5  alpha_opt=0.01   alpha=0.01   v=0.001    abs_diff=5.604e-13
v_opt=0.5   c=0.5  alpha_opt=0.01   alpha=0.01   v=0.1      abs_diff=2.001e-12
v_opt=0.5   c=0.5  alpha_opt=0.01   alpha=0.01   v=1        abs_diff=1.819e-12
v_opt=0.5   c=0.5  alpha_opt=0.01   alpha=0.01   v=10       abs_diff=4.547e-12
v_opt=0.5   c=0.5  alpha_opt=0.01   alpha=0.01   v=100      abs_diff=1.137e-11
v_opt=5     c=1    alpha_opt=0.025  alpha=0.025  v=1e-09    abs_diff=9.006e-13
v_opt=5     c=1    alpha_opt=0.025  alpha=0.025  v=1e-06    abs_diff=1.322e-12
v_opt=5     c=1    alpha_opt=0.025  alpha=0.025  v=0.001    abs_diff=2.315e-12
v_opt=5     c=1    alpha_opt=0.025  alpha=0.025  v=0.1      abs_diff=2.548e-12
v_opt=5     c=1    alpha_opt=0.025  alpha=0.025  v=1        abs_diff=1.819e-12
v_opt=5     c=1    alpha_opt=0.025  alpha=0.025  v=1

In [6]:
rng = np.random.default_rng(0)

random_rows = []
for _ in range(50):
    v_opt = float(10 ** rng.uniform(-2, 2))
    c = float(10 ** rng.uniform(-1, 1))
    alpha_opt = float(10 ** rng.uniform(-3, np.log10(0.1)))
    alpha = float(10 ** rng.uniform(-4, np.log10(0.2)))
    v = float(10 ** rng.uniform(-9, 3))

    local_bound = GammaExponentialMixtureBound(v_opt=v_opt, c=c, alpha_opt=alpha_opt)
    mapie_value = local_bound.bound(v=v, alpha=alpha)
    confseq_value = gamma_exponential_mixture_bound(
        v=v,
        alpha=alpha,
        v_opt=v_opt,
        c=c,
        alpha_opt=alpha_opt,
    )

    random_rows.append(
        {
            "v_opt": v_opt,
            "c": c,
            "alpha_opt": alpha_opt,
            "alpha": alpha,
            "v": v,
            "mapie": mapie_value,
            "confseq": confseq_value,
            "abs_diff": abs(mapie_value - confseq_value),
        }
    )

worst_random_case = max(random_rows, key=lambda row: row["abs_diff"])
print("max_random_abs_diff =", worst_random_case["abs_diff"])
print("worst_random_case =", worst_random_case)

max_random_abs_diff = 6.900791049702093e-11
worst_random_case = {'v_opt': 0.05977493211406724, 'c': 1.1273129532463715, 'alpha_opt': 0.011139596365128384, 'alpha': 0.00019659805464552026, 'v': 607.1728134641914, 'mapie': 151.8700394113311, 'confseq': 151.8700394114001, 'abs_diff': 6.900791049702093e-11}


In [7]:
local_v0 = GammaExponentialMixtureBound(v_opt=1.0, c=1.0, alpha_opt=0.05).bound(
    v=0.0,
    alpha=0.05,
)
print("MAPIE v=0 bound:", local_v0)

try:
    confseq_v0 = gamma_exponential_mixture_bound(
        v=0.0,
        alpha=0.05,
        v_opt=1.0,
        c=1.0,
        alpha_opt=0.05,
    )
    print("confseq v=0 bound:", confseq_v0)
except Exception as exc:
    print("confseq v=0 raised:", type(exc).__name__, str(exc))

MAPIE v=0 bound: 3.2419825312017565
confseq v=0 raised: RuntimeError Failed to find an upper limit for the mixture bound
